In [1]:
import numpy as np
import sympy as sp
from typing import Any

#Defining all the Quantum States
KetH: sp.Matrix = sp.Matrix([
    [1],
    [0]
])

KetV: sp.Matrix = sp.Matrix([
    [0],
    [1]
])

KetD: sp.Matrix = (sp.sqrt(2)**-1) *sp.Matrix([
    [1],
    [1]
])

KetA: sp.Matrix = (sp.sqrt(2)**-1) * sp.Matrix([
    [1],
    [-1]
])

KetR: sp.Matrix = (sp.sqrt(2)**-1) * sp.Matrix([
    [1],
    [sp.I]
])

KetL: sp.Matrix = (sp.sqrt(2)**-1) * sp.Matrix([
    [1],
    [-sp.I]
])

#Defining the gates

X: sp.Matrix = sp.Matrix([
    [0, 1],
    [1, 0]
])

Y: sp.Matrix = sp.Matrix([
    [0, -sp.I],
    [sp.I, 0]
])

Z: sp.Matrix = sp.Matrix([
    [1, 0],
    [0, -1]
])

H: sp.Matrix = (sp.sqrt(2) ** -1) * sp.Matrix([
    [1, 1],
    [1, -1]
])

Q: sp.Matrix = (sp.sqrt(2) ** -1) * sp.Matrix([
    [1, 1],
    [sp.I, -sp.I]
])

S: sp.Matrix = sp.Matrix([
    [1, 0],
    [0, sp.I]
])

ST: sp.Matrix = sp.Matrix([
    [1, 0],
    [0, -sp.I]
])

T: sp.Matrix = sp.Matrix([
    [1, 0],
    [0, sp.E**(sp.I * sp.pi / 4)]
])

TT: sp.Matrix = sp.Matrix([
    [1, 0],
    [0, sp.E**(-sp.I * sp.pi / 4)]
])

P0: sp.Matrix = sp.Matrix([
    [1, 0],
    [0, 0]
])

P1: sp.Matrix = sp.Matrix([
    [0, 0],
    [0, 1]
])

C: sp.Matrix = sp.Matrix([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 0, 1],
    [0, 0, 1, 0]
])


#Transforms Ket to a Bra
def bra(ket):
    return ket.conjugate().T


#calculate probability using born's rule
def probability(observed, psi):
    innerprod: Any = bra(observed) * psi
    return (innerprod * innerprod.conjugate())[0]

def hermitian_conjugate(A: sp.Matrix) -> sp.Matrix:
    return A.T.conjugate()

In [100]:
#Helper Functions
def isSymmetric(A: sp.Matrix):
    return A.T == A

def isUnitary(A: sp.Matrix):
    return hermitian_conjugate(A) == A.inv()

def isHermitian(A: sp.Matrix):
    return hermitian_conjugate(A) == A

def isInvolution(A: sp.Matrix):
    return A == A.inv()

def isOrthogonal(A: sp.Matrix):
    return A.T == A.inv()

def expectationValue(O: sp.Matrix, psi: sp.Matrix):
    psi_norm = psi.normalized()
    return sp.simplify((psi_norm.H * O * psi_norm)[0])

def calculateObservable(A: sp.Matrix, alpha, beta):
    eigenVects = A.eigenvects()
    psi = sp.Matrix([alpha, beta])

    #extract eigenvectors and create the Unitary Matrix
    mapping = []
    for val, mult, vects in eigenVects:
        mapping.append((val, vects[0].normalized()))

    
    U = sp.Matrix.hstack(*[vec for val, vec in mapping])

    state_rotated = U.H * psi
    results = {}
    for i, (val, vec) in enumerate(mapping):
        prob = sp.simplify(sp.Abs(state_rotated[i])**2)
        results[val] = prob

    return results

def U3(theta, phi, lamb):
    return sp.Matrix([
        [sp.cos(theta/2), -sp.exp(sp.I*lamb) * sp.sin(theta/2)],
        [sp.exp(sp.I*phi)*sp.sin(theta/2), sp.exp(sp.I * (lamb + phi))*sp.cos(theta/2)]
    ])

def execute_circuit(initial_state, gates):
    psi = gates[len(gates) - 1]
    for i in reversed(range(0, len(gates) - 1)):
        psi = psi @ gates[i]
    psi = psi @ initial_state
    return sp.simplify(psi)

def express_in_basis(psi, basis_kets):
    #writes psi as alpha * Ket1 + beta * KetB and returns alpha, beta
    c1 = bra(basis_kets[0]) * psi
    c2 = bra(basis_kets[1]) * psi
    return c1, c2

def change_basis(target_state, basis_vectors):
    M = sp.Matrix.hstack(*basis_vectors)
    c = M.solve(target_state)
    return c

In [107]:
#Work area
X @ Y @ KetH

#Conjugate Transpose
S.H
Y.H

#Transpose
print(Y.T)

print(hermitian_conjugate(Y))
print(isSymmetric(S))

#example of calculating observables
A = (sp.sqrt(2) ** -1) * sp.Matrix([[0, 1 - sp.I],[sp.I+1, 0]])
alpha = -1/sp.sqrt(2)
beta = (1+sp.I)/2
results = calculateObservable(A, alpha, beta)
print(f"probability of observing 1: {results[1]}\nprobability of observing -1: {results[-1]}")

#Example representing R or L as DA basis
coeffs = change_basis(KetR, [KetD, KetA])
alpha = coeffs[0]
beta = coeffs[1]
print(f"Alpha: {alpha}, Beta: {beta}")

psi = sp.Matrix([[-sp.I], [sp.exp(sp.I * sp.pi / 4)]])
num = complex(expectationValue(Z, psi)) * 1.0
print(num)


#finding expectation values
A = 1/7 * sp.Matrix([[5, -2], [-2, 2]])
print(A.eigenvals())
print(A.eigenvects())
alpha, beta = (sp.sqrt(5) ** -1) * sp.Matrix([[-2], [1]]), (sp.sqrt(5)**-1) * sp.Matrix([[1], [2]])
print(probability(alpha, KetH))
print(probability(beta, KetH))
print(expectationValue(A, KetH))

# Changing psi to computational basis and R/L basis
c1 = 1 / (2 * sp.sqrt(2))
c2 = (sp.I * sp.sqrt(7)) / (2 * sp.sqrt(2))
psi_11 = c1 * KetD - c2 * KetA
sp.simplify(express_in_basis(psi_11, [KetH, KetV]))

# Gate sequences
sp.simplify(execute_circuit(KetH, [H, ST, X, T]))
T @ X @ ST @ H @ KetH

result = T.T.conjugate() == T.inv()
print(result)
T @ T.T.conjugate()

U3(2*sp.pi/3, sp.pi/2, 0) @ KetH

A = sp.Matrix([[2, 1+ sp.I], [1 - sp.I, 3]])
phi1 = (sp.sqrt(6)**-1) * sp.Matrix([[1+sp.I], [2]])
phi2 = (sp.sqrt(3)**-1) * sp.Matrix([[1+sp.I], [-1]])
sp.simplify(A * phi1)
psi = sp.cos(sp.pi/6) * phi1 + sp.exp(-sp.I * sp.pi / 3) * sp.sin(sp.pi/6) *phi2
print(expectationValue(A, psi))

coeffs = change_basis(KetA, [KetR, KetL])
alpha = coeffs[0]
beta = coeffs[1]
print(f"Alpha: {alpha}, Beta: {beta}")

ST @ H @ KetH

print(T.T.conjugate() == T.inv())
U3(2*sp.pi/3, sp.pi/2, 0)

A = sp.Matrix([[2, 1+ sp.I], [1 - sp.I, 3]])
print(A.T.conjugate() == A)

Matrix([[0, I], [-I, 0]])
Matrix([[0, -I], [I, 0]])
True
probability of observing 1: 0
probability of observing -1: 1
Alpha: 1/2 + I/2, Beta: 1/2 - I/2
0j
{0.857142857142857: 1, 0.142857142857143: 1}
[(0.857142857142857, 1, [Matrix([
[ 0.894427190999916],
[-0.447213595499958]])]), (0.142857142857143, 1, [Matrix([
[0.447213595499958],
[0.894427190999916]])])]
4/5
1/5
0.714285714285714
True
13/4
Alpha: 1/2 + I/2, Beta: 1/2 - I/2
True
True


## Wave Plate Functions (Polarization Optics)

These functions model optical elements used to transform polarization states of light using Jones calculus.

- **`HWP(theta)`** — Half-wave plate Jones matrix. Use when a problem says "light passes through a HWP with fast axis at angle θ". The HWP flips the polarization about the fast axis — it converts |H⟩ to an arbitrary linear polarization. *Use for: HW1-type problems, any problem involving linear polarization rotation.*

- **`QWP(phi)`** — Quarter-wave plate Jones matrix. Use when light passes through a QWP with fast axis at angle φ. The QWP converts linear polarization to elliptical/circular polarization. *Use for: Creating circular polarization states, HW1 Problem 3.*

- **`prepare_state_HWP(theta)`** — Shortcut for HWP(θ)|H⟩. Use when you need the output state after horizontally polarized light passes through a HWP. Returns the resulting polarization state directly. *Use for: Quick polarization state preparation.*

- **`prepare_state_QWP_HWP(phi, theta)`** — Shortcut for QWP(φ)·HWP(θ)|H⟩. Use when light passes through a HWP then a QWP. This is the general method to prepare any polarization state on the Poincaré sphere. *Use for: HW1 Problem 3, preparing arbitrary polarization states.*

In [4]:
def HWP(theta):
    """Half-wave plate Jones matrix.
    
    Parameters:
        theta: Fast axis angle in radians.
    
    Returns:
        2x2 sympy Matrix representing the HWP transformation.
    """
    return sp.Matrix([
        [sp.cos(2*theta), sp.sin(2*theta)],
        [sp.sin(2*theta), -sp.cos(2*theta)]
    ])

def QWP(phi):
    """Quarter-wave plate Jones matrix.
    
    Parameters:
        phi: Fast axis angle in radians.
    
    Returns:
        2x2 sympy Matrix representing the QWP transformation.
    """
    prefactor = (1 - sp.I) / 2
    return sp.simplify(prefactor * sp.Matrix([
        [1 + sp.I*sp.cos(2*phi), sp.I*sp.sin(2*phi)],
        [sp.I*sp.sin(2*phi), 1 - sp.I*sp.cos(2*phi)]
    ]))

def prepare_state_HWP(theta):
    """Returns the state HWP(theta)|H⟩ = [[cos(2θ)], [sin(2θ)]].
    
    Parameters:
        theta: HWP fast axis angle in radians.
    
    Returns:
        2x1 sympy Matrix (ket vector).
    """
    return sp.simplify(sp.Matrix([
        [sp.cos(2*theta)],
        [sp.sin(2*theta)]
    ]))

def prepare_state_QWP_HWP(phi, theta):
    """Returns QWP(phi) * HWP(theta) * |H⟩ using the course formula.
    
    Parameters:
        phi: QWP fast axis angle in radians.
        theta: HWP fast axis angle in radians.
    
    Returns:
        2x1 sympy Matrix (ket vector).
    """
    prefactor = (1 - sp.I) / 2
    return sp.simplify(prefactor * sp.Matrix([
        [sp.cos(2*(phi - theta)) + sp.I*sp.cos(2*theta)],
        [sp.sin(2*(phi - theta)) + sp.I*sp.sin(2*theta)]
    ]))

In [75]:
A_ARR = sp.sqrt(2) * T * X * TT
A_ARR

val = (ST @ H @ KetH == KetL)
print(val)

True


In [70]:
HWP(5)
prepare_state_QWP_HWP(0, 0)

Matrix([
[1],
[0]])

## Inner Product & Basis Functions

These functions handle inner products, normalization checks, and basis validation.

- **`inner_product(phi, psi)`** — Computes the inner product ⟨φ|ψ⟩ as a simplified scalar. Handles the bra conversion internally so you just pass two kets. *Use for: Any problem asking for overlap between states, transition amplitudes, orthogonality checks.*

- **`is_normalized(ket)`** — Checks if ⟨ψ|ψ⟩ = 1. Returns True/False. *Use for: Verifying that a state vector is a valid quantum state before using it in calculations.*

- **`is_orthonormal_basis(ket_a, ket_b)`** — Checks all three conditions for a valid orthonormal basis: both kets normalized and mutually orthogonal. Returns a detailed dict. *Use for: HW2 Problem 2, any problem asking "do these states form a basis?"*

- **`outer_product(ket_a, ket_b)`** — Computes |a⟩⟨b| as a matrix. *Use for: Constructing operators, density matrices, projectors, spectral decompositions.*

- **`projector(ket)`** — Returns the projection operator |ψ⟩⟨ψ| onto the given state. *Use for: Constructing measurement operators, spectral decomposition terms, Born rule calculations.*

In [5]:
def inner_product(phi, psi):
    """Computes the inner product ⟨φ|ψ⟩ and returns a simplified scalar.
    
    Parameters:
        phi: Column vector (ket) for the bra side.
        psi: Column vector (ket) for the ket side.
    
    Returns:
        Simplified scalar value of ⟨φ|ψ⟩.
    """
    return sp.simplify((bra(phi) * psi)[0])

def is_normalized(ket):
    """Checks if a state vector is normalized (⟨ψ|ψ⟩ = 1).
    
    Parameters:
        ket: Column vector (state) to check.
    
    Returns:
        True if the state is normalized, False otherwise.
    """
    return sp.simplify(inner_product(ket, ket) - 1) == 0

def is_orthonormal_basis(ket_a, ket_b):
    """Checks if two kets form an orthonormal basis.
    
    Verifies three conditions:
    1. ⟨a|a⟩ = 1 (ket_a is normalized)
    2. ⟨b|b⟩ = 1 (ket_b is normalized)
    3. ⟨a|b⟩ = 0 (mutual orthogonality)
    
    Parameters:
        ket_a: First basis ket.
        ket_b: Second basis ket.
    
    Returns:
        Dict with keys 'a_normalized', 'b_normalized', 'orthogonal', 'is_basis'.
    """
    a_norm = is_normalized(ket_a)
    b_norm = is_normalized(ket_b)
    ortho = sp.simplify(inner_product(ket_a, ket_b)) == 0
    return {
        "a_normalized": a_norm,
        "b_normalized": b_norm,
        "orthogonal": ortho,
        "is_basis": a_norm and b_norm and ortho
    }

def outer_product(ket_a, ket_b):
    """Computes the outer product |a⟩⟨b|.
    
    Parameters:
        ket_a: Column vector for the ket side.
        ket_b: Column vector for the bra side.
    
    Returns:
        Matrix representing |a⟩⟨b|.
    """
    return sp.simplify(ket_a * bra(ket_b))

def projector(ket):
    """Returns the projection operator |ψ⟩⟨ψ| onto the given state.
    
    Parameters:
        ket: Column vector (state) to project onto.
    
    Returns:
        Matrix representing the projector |ψ⟩⟨ψ|.
    """
    return sp.simplify(ket * bra(ket))

## Measurement & Observable Functions

These functions handle quantum measurements, spectral decomposition, and Bloch sphere representation.

- **`measurement_probabilities(psi, observable)`** — Given a state and a Hermitian observable, performs eigendecomposition and returns a dict mapping each eigenvalue to its measurement probability via Born rule. *Use for: HW2 Problem 6, any "what is the probability of measuring eigenvalue aₖ?" question.*

- **`bloch_vector(psi)`** — Computes the Bloch vector (⟨X⟩, ⟨Y⟩, ⟨Z⟩) for a single-qubit state. These are the Cartesian coordinates on the Bloch sphere. *Use for: Big Practice Problem 9, visualizing where a state sits on the Bloch sphere.*

- **`spectral_decomposition(A)`** — Takes a Hermitian matrix and returns its spectral decomposition as a list of (eigenvalue, eigenvector, projector) tuples. Verifies that Σ λₖPₖ = A. *Use for: Big Practice Problems 3/10, expressing an observable in terms of its projectors.*

- **`verify_eigenpair(A, eigenvalue, eigenvector)`** — Checks whether A|v⟩ = λ|v⟩ holds. *Use for: Big Practice Problem 10, verifying hand-computed eigenvalues/eigenvectors.*

- **`active_measurement_matrix(phi0, phi1)`** — Given two orthonormal basis kets, constructs the unitary U whose rows are the bras. Apply U before a Z-measurement to effectively measure in the {|φ₀⟩, |φ₁⟩} basis. *Use for: HW2 Problem 5, measuring in an arbitrary basis.*

In [6]:
def measurement_probabilities(psi, observable):
    """Computes measurement probabilities for an observable on a given state.
    
    Performs eigendecomposition of the observable, then uses Born rule
    P(aₖ) = |⟨φₖ|ψ⟩|² to compute the probability of each outcome.
    
    Parameters:
        psi: Column vector (state) being measured.
        observable: Hermitian matrix (observable being measured).
    
    Returns:
        Dict mapping eigenvalues to their measurement probabilities.
    """
    eigenvects = observable.eigenvects()
    results = {}
    for val, mult, vects in eigenvects:
        vec = vects[0].normalized()
        prob = sp.simplify(sp.Abs(inner_product(vec, psi))**2)
        results[val] = prob
    return results

def bloch_vector(psi):
    """Computes the Bloch vector (⟨X⟩, ⟨Y⟩, ⟨Z⟩) for a single-qubit state.
    
    The Bloch vector gives the Cartesian coordinates on the Bloch sphere.
    For a pure state: r = (⟨σx⟩, ⟨σy⟩, ⟨σz⟩).
    
    Parameters:
        psi: 2x1 column vector (single-qubit state).
    
    Returns:
        Tuple of three simplified values (⟨X⟩, ⟨Y⟩, ⟨Z⟩).
    """
    ex = sp.simplify(expectationValue(X, psi))
    ey = sp.simplify(expectationValue(Y, psi))
    ez = sp.simplify(expectationValue(Z, psi))
    return (ex, ey, ez)

def spectral_decomposition(A):
    """Computes the spectral decomposition of a Hermitian matrix.
    
    Returns eigenvalues with their eigenvectors and projectors, and verifies
    that A = Σ λₖ|φₖ⟩⟨φₖ|.
    
    Parameters:
        A: Hermitian matrix to decompose.
    
    Returns:
        List of tuples [(eigenvalue, eigenvector, projector), ...].
    """
    eigenvects = A.eigenvects()
    decomp = []
    reconstructed = sp.zeros(*A.shape)
    for val, mult, vects in eigenvects:
        vec = vects[0].normalized()
        proj = projector(vec)
        decomp.append((val, vec, proj))
        reconstructed += val * proj
    
    # Verify reconstruction
    diff = sp.simplify(reconstructed - A)
    if diff != sp.zeros(*A.shape):
        print("Warning: reconstruction check failed. Difference:", diff)
    
    return decomp

def verify_eigenpair(A, eigenvalue, eigenvector):
    """Checks whether A|v⟩ = λ|v⟩.
    
    Parameters:
        A: Square matrix.
        eigenvalue: Proposed eigenvalue λ.
        eigenvector: Proposed eigenvector |v⟩ (column vector).
    
    Returns:
        True if A|v⟩ = λ|v⟩, False otherwise.
    """
    lhs = sp.simplify(A * eigenvector)
    rhs = sp.simplify(eigenvalue * eigenvector)
    return sp.simplify(lhs - rhs) == sp.zeros(eigenvector.rows, 1)

def active_measurement_matrix(phi0, phi1):
    """Constructs the unitary for measurement in the {|φ₀⟩, |φ₁⟩} basis.
    
    The resulting matrix U has rows ⟨φ₀| and ⟨φ₁|. Applying U before a
    standard Z-measurement is equivalent to measuring in the given basis.
    
    Parameters:
        phi0: First basis ket |φ₀⟩.
        phi1: Second basis ket |φ₁⟩.
    
    Returns:
        2x2 unitary matrix.
    """
    return sp.simplify(sp.Matrix([
        [bra(phi0)],
        [bra(phi1)]
    ]))

## Gate & Transformation Functions

These functions handle gate algebra, similarity transforms, and U3 parameter extraction.

- **`similarity_transform(U, A)`** — Computes U·A·U†. This is how you express an operator in a different basis or compute the effect of conjugation by a gate. *Use for: HW2 Problem 6 (computing √2·TXT†), any "compute UAU†" problem.*

- **`verify_gate_identity(gate_list, target)`** — Multiplies a list of gates left-to-right and checks if the result equals a target matrix. *Use for: HW2 Problem 3, verifying circuit identities like HXH = Z or HZH = X.*

- **`find_U3_parameters(target_matrix)`** — Given a 2×2 unitary, extracts the θ, φ, λ parameters from the U3 decomposition. *Use for: Big Practice Problems 8/9, expressing any single-qubit gate in U3 form.*

- **`state_from_bloch(theta, phi)`** — Creates cos(θ/2)|0⟩ + e^(iφ)sin(θ/2)|1⟩ from Bloch sphere angles. *Use for: Converting Bloch sphere coordinates to a state vector, creating states at known positions on the sphere.*

In [7]:
def similarity_transform(U, A):
    """Computes the similarity transform U·A·U†.
    
    Parameters:
        U: Unitary matrix.
        A: Matrix to transform.
    
    Returns:
        Simplified matrix U·A·U†.
    """
    return sp.simplify(U * A * hermitian_conjugate(U))

def verify_gate_identity(gate_list, target):
    """Multiplies gates left-to-right and checks if the product equals a target.
    
    Parameters:
        gate_list: List of matrices to multiply in order.
        target: Target matrix to compare against.
    
    Returns:
        True if the product equals the target, False otherwise.
    """
    product = gate_list[0]
    for g in gate_list[1:]:
        product = product * g
    product = sp.simplify(product)
    print("Product:", product)
    return sp.simplify(product - target) == sp.zeros(*target.shape)

def find_U3_parameters(target_matrix):
    """Extracts θ, φ, λ parameters from a 2×2 unitary matrix.
    
    Matches entries to the U3 formula:
        U3 = [[cos(θ/2), -e^(iλ)sin(θ/2)],
              [e^(iφ)sin(θ/2), e^(i(λ+φ))cos(θ/2)]]
    
    Parameters:
        target_matrix: 2×2 unitary matrix.
    
    Returns:
        Dict with keys 'theta', 'phi', 'lambda' containing symbolic values.
    """
    a = sp.simplify(target_matrix[0, 0])
    b = sp.simplify(target_matrix[0, 1])
    c = sp.simplify(target_matrix[1, 0])
    
    # θ from cos(θ/2) = |a|
    theta = sp.simplify(2 * sp.acos(sp.Abs(a)))
    
    # Handle edge cases
    sin_half = sp.simplify(sp.sin(theta / 2))
    
    if sin_half == 0:
        # θ = 0, gate is a global phase; φ and λ are degenerate
        phase = sp.arg(a)
        return {"theta": sp.S(0), "phi": sp.S(0), "lambda": sp.simplify(phase)}
    
    # φ from e^(iφ)·sin(θ/2) = c  =>  φ = arg(c)
    phi = sp.simplify(sp.arg(c))
    
    # λ from -e^(iλ)·sin(θ/2) = b  =>  e^(iλ) = -b/sin(θ/2)  =>  λ = arg(-b)
    lamb = sp.simplify(sp.arg(-b))
    
    return {"theta": sp.simplify(theta), "phi": phi, "lambda": lamb}

def state_from_bloch(theta, phi):
    """Creates a state vector from Bloch sphere angles.
    
    Returns cos(θ/2)|0⟩ + e^(iφ)sin(θ/2)|1⟩.
    
    Parameters:
        theta: Polar angle (0 to π).
        phi: Azimuthal angle (0 to 2π).
    
    Returns:
        2x1 sympy Matrix (ket vector).
    """
    return sp.simplify(sp.Matrix([
        [sp.cos(theta/2)],
        [sp.exp(sp.I*phi) * sp.sin(theta/2)]
    ]))

## Display & Utility Functions

General-purpose helper functions for display, algebraic operations, and quick lookups.

- **`pretty_print(psi)`** — Displays a 2×1 state vector in Dirac notation like "α|0⟩ + β|1⟩". *Use for: Making output human-readable when checking answers.*

- **`commutator(A, B)`** — Computes [A,B] = AB - BA. Two observables commute iff [A,B] = 0, meaning they can be simultaneously measured. *Use for: Checking if two observables are compatible, verifying algebraic identities.*

- **`anticommutator(A, B)`** — Computes {A,B} = AB + BA. The Pauli matrices satisfy {σᵢ,σⱼ} = 2δᵢⱼI. *Use for: Verifying Pauli algebra identities, Clifford algebra problems.*

- **`malus_probability(theta_degrees)`** — Given a HWP angle in degrees, returns (P_H, P_V) = (cos²(2θ), sin²(2θ)). Quick lookup for polarization measurement probabilities. *Use for: Fast probability calculations for HWP problems when you just need a number.*

In [8]:
def pretty_print(psi):
    """Displays a 2×1 state vector in Dirac notation.
    
    Parameters:
        psi: 2×1 column vector (ket).
    
    Returns:
        String in Dirac notation, e.g., "(1/√2)|0⟩ + (i/√2)|1⟩".
    """
    alpha = sp.nsimplify(psi[0], rational=False)
    beta = sp.nsimplify(psi[1], rational=False)
    
    parts = []
    if alpha != 0:
        if alpha == 1:
            parts.append("|0⟩")
        elif alpha == -1:
            parts.append("-|0⟩")
        else:
            parts.append(f"({sp.pretty(alpha)})|0⟩")
    
    if beta != 0:
        if beta == 1:
            parts.append("+ |1⟩" if parts else "|1⟩")
        elif beta == -1:
            parts.append("- |1⟩" if parts else "-|1⟩")
        else:
            beta_str = sp.pretty(beta)
            if parts:
                # Check if beta is negative/has leading minus
                if str(beta).startswith("-"):
                    parts.append(f"- ({sp.pretty(-beta)})|1⟩")
                else:
                    parts.append(f"+ ({beta_str})|1⟩")
            else:
                parts.append(f"({beta_str})|1⟩")
    
    result = " ".join(parts) if parts else "0"
    print(result)
    return result

def commutator(A, B):
    """Computes the commutator [A, B] = AB - BA.
    
    Parameters:
        A: Square matrix.
        B: Square matrix of same dimensions.
    
    Returns:
        Simplified matrix [A, B].
    """
    return sp.simplify(A * B - B * A)

def anticommutator(A, B):
    """Computes the anticommutator {A, B} = AB + BA.
    
    Parameters:
        A: Square matrix.
        B: Square matrix of same dimensions.
    
    Returns:
        Simplified matrix {A, B}.
    """
    return sp.simplify(A * B + B * A)

def malus_probability(theta_degrees):
    """Computes HWP polarization measurement probabilities.
    
    Given a HWP fast-axis angle in degrees, returns the probabilities
    of measuring H and V polarization: P_H = cos²(2θ), P_V = sin²(2θ).
    
    Parameters:
        theta_degrees: HWP fast-axis angle in degrees.
    
    Returns:
        Tuple (P_H, P_V) of simplified probabilities.
    """
    theta = sp.rad(theta_degrees)
    p_h = sp.simplify(sp.cos(2*theta)**2)
    p_v = sp.simplify(sp.sin(2*theta)**2)
    return (p_h, p_v)

## Tests & Demonstrations

In [9]:
# === TESTS ===

# Test 1: HWP at 22.5° should produce |D⟩
print("Test 1: HWP(22.5°)|H⟩ =", sp.simplify(prepare_state_HWP(sp.pi/8)))

# Test 2: Z gate bug fix verification
assert Z == sp.Matrix([[1,0],[0,-1]]), "Z gate is still wrong!"
print("Test 2: Z gate is correct ✓")

# Test 3: U3 should work without errors
print("Test 3: U3(π/2, 0, π) =", U3(sp.pi/2, 0, sp.pi))  # Should give Hadamard

# Test 4: Orthonormality check
print("\nTest 4a: Is {|D⟩, |A⟩} a basis?", is_orthonormal_basis(KetD, KetA))
print("Test 4b: Is {|D⟩, |R⟩} a basis?", is_orthonormal_basis(KetD, KetR))

# Test 5: Bloch vector for |D⟩ (should be (1, 0, 0))
print("\nTest 5: Bloch vector of |D⟩:", bloch_vector(KetD))

# Test 6: Measurement probabilities
print("\nTest 6: Measuring |R⟩ with Z:", measurement_probabilities(KetR, Z))

# Test 7: Similarity transform TXT†
A_test = sp.sqrt(2) * similarity_transform(T, X)
print("\nTest 7: √2 TXT† =", sp.simplify(A_test))

# Test 8: Verify HXH = Z
print("\nTest 8: HXH = Z?", verify_gate_identity([H, X, H], Z))

# Test 9: Spectral decomposition of X
print("\nTest 9: Spectral decomposition of X:")
for val, vec, proj in spectral_decomposition(X):
    print(f"  eigenvalue={val}, eigenvector={vec.T}, projector=\n{proj}")

# Test 10: QWP(0)HWP(0)|H⟩ should give |H⟩
print("\nTest 10: QWP(0)HWP(0)|H⟩ =", sp.simplify(prepare_state_QWP_HWP(0, 0)))

# Test 11: Active measurement matrix for circular basis
print("\nTest 11: Circular-basis measurement gate:")
print(active_measurement_matrix(KetR, KetL))

# Test 12: find_U3_parameters for Hadamard
print("\nTest 12: U3 params for H:", find_U3_parameters(H))

# Test 13: Pretty print
print("\nTest 13: Pretty print |R⟩:")
pretty_print(KetR)

# Test 14: Commutator [X, Y] = 2iZ
print("\nTest 14: [X, Y] =", commutator(X, Y))
print("Expected: 2iZ =", 2*sp.I*Z)

# Test 15: Malus probability at 30°
print("\nTest 15: Malus prob at 30°:", malus_probability(30))

# Test 16: State from Bloch sphere (north pole = |0⟩)
print("\nTest 16: state_from_bloch(0, 0) =", state_from_bloch(0, 0).T)
print("         state_from_bloch(π, 0) =", state_from_bloch(sp.pi, 0).T)

# Test 17: KetH naming fix
print("\nTest 17: KetH is defined:", KetH.T)

Test 1: HWP(22.5°)|H⟩ = Matrix([[sqrt(2)/2], [sqrt(2)/2]])
Test 2: Z gate is correct ✓
Test 3: U3(π/2, 0, π) = Matrix([[sqrt(2)/2, sqrt(2)/2], [sqrt(2)/2, -sqrt(2)/2]])

Test 4a: Is {|D⟩, |A⟩} a basis? {'a_normalized': True, 'b_normalized': True, 'orthogonal': True, 'is_basis': True}
Test 4b: Is {|D⟩, |R⟩} a basis? {'a_normalized': True, 'b_normalized': True, 'orthogonal': False, 'is_basis': False}

Test 5: Bloch vector of |D⟩: (1, 0, 0)

Test 6: Measuring |R⟩ with Z: {-1: 1/2, 1: 1/2}

Test 7: √2 TXT† = Matrix([[0, -(-1)**(3/4)*sqrt(2)], [(-1)**(1/4)*sqrt(2), 0]])
Product: Matrix([[1, 0], [0, -1]])

Test 8: HXH = Z? True

Test 9: Spectral decomposition of X:
  eigenvalue=-1, eigenvector=Matrix([[-sqrt(2)/2, sqrt(2)/2]]), projector=
Matrix([[1/2, -1/2], [-1/2, 1/2]])
  eigenvalue=1, eigenvector=Matrix([[sqrt(2)/2, sqrt(2)/2]]), projector=
Matrix([[1/2, 1/2], [1/2, 1/2]])

Test 10: QWP(0)HWP(0)|H⟩ = Matrix([[1], [0]])

Test 11: Circular-basis measurement gate:
Matrix([[sqrt(2)/2, -sqrt(